# 🧠 Student Mental-Health Digital Twin — 8-Layer Walkthrough

Cells mirror the project: Setup → Load → Preprocess → Train (best-F1) → Twin → What-if → Risk → Recommendations → Charts → Report.
Run top-to-bottom. Needs `Depression_Type` CSV at `../data/Mental Health Classification.csv`, else synthetic fallback runs.

In [ ]:
# Cell 1 - Setup
%pip install -q pandas numpy scikit-learn matplotlib joblib xgboost
import os, sys
sys.path.insert(0, '..')
from src import *
from src.config import FEATURES, DEPRESSION_TYPES, RISK_LEVELS
print(f'{len(FEATURES)} features | {len(DEPRESSION_TYPES)} labels | {RISK_LEVELS}')

In [ ]:
# Cell 2 — Config constants
from src.config import FEATURES, DEPRESSION_TYPES, risk_level_from_score
print('FEATURES:', FEATURES)
print('LABELS:', DEPRESSION_TYPES)
print('score 7 ->', risk_level_from_score(7), '| score 25 ->', risk_level_from_score(25))

In [ ]:
# Cell 3 — Load Data (real CSV or synthetic fallback)
df, used_real = load_data()
print('real_data =', used_real, df.shape)
df.head(3)

In [ ]:
# Cell 4 — Layer 1 Preprocessing: scale + split
X, y = prepare_features(df)
from sklearn.preprocessing import LabelEncoder
label_names = list(LabelEncoder().fit(df['Depression_Type'].astype(str)).classes_)
Xtr, Xte, ytr, yte, scaler, cols = split_and_scale(X, y)
print('train', Xtr.shape, 'test', Xte.shape, '| classes:', len(label_names))

In [ ]:
# Cell 5 — Layer 2 Predictive Intelligence: 4 models, best by F1
best_name, best, fitted, results = train_and_select(Xtr, Xte, ytr, yte, label_names)
print('BEST =', best_name)

In [ ]:
# Cell 6 - Layer 3 Digital Twin Engine (live mirror + history)
state = {cc: float(X[cc].median()) for cc in cols}
row0 = X.iloc[0].to_dict()
state.update({k: float(v) for k, v in row0.items()})
twin = StudentDigitalTwin('STU-001', state, model=best, scaler=scaler, feature_columns=cols, label_names=label_names)
print(twin)
twin.predict()

In [ ]:
# Cell 7 - Layer 4 Scenario Analysis (what-if, then restore)
outs = compare_scenarios(twin)
for o in outs:
    print(o['scenario'], o['before']['score'], '->', o['after']['score'], 'delta', o['delta_score'])

In [ ]:
# Cell 8 - Layer 5 Risk Scoring (0-30, level, top-3, trend)
pred = twin.predict()
risk = risk_profile(pred['proba'], twin.state, label_names)
print(risk)
sleep_col = 'Sleep_Hours' if 'Sleep_Hours' in twin.state else cols[0]
twin.update(**{sleep_col: float(df[sleep_col].min())})
s1 = risk_profile(twin.predict()['proba'], twin.state, label_names)['score']
twin.update(**{sleep_col: float(df[sleep_col].max())})
s2 = risk_profile(twin.predict()['proba'], twin.state, label_names)['score']
print('trend demo:', [risk['score'], s1, s2], '->', risk_trend([risk['score'], s1, s2]))

In [ ]:
# Cell 9 — Layer 6 Recommendation Engine (rule-based)
reco = recommend(risk, twin.state)
print(reco['risk_level'], reco['risk_score'], reco['flags'])
for s in reco['strategies']: print('•', s)
print('Roadmap:', reco['weekly_roadmap'])
print('Say:', reco['encouragement'])
print(reco['disclaimer'])

In [ ]:
# Cell 10 — Layer 7 Visualization
from src.visualization import plot_history, plot_top3, plot_model_comparison
plot_model_comparison(results); plot_top3(risk); plot_history(twin.history_df())

In [ ]:
# Cell 11 — Layer 8 Persistence & Report
save_bundle('../models/bundle.joblib', best, scaler, cols, label_names, results)
export_history_csv(twin, '../reports/history_STU-001.csv')
write_report('../reports/report_STU-001.txt', 'STU-001', risk, reco, outs)
print('saved bundle + csv + report')
print(open('../reports/report_STU-001.txt', encoding='utf-8').read()[:1200])